# Give Me Some Credit - EDA

Purpose: explore the credit-risk dataset, inspect missing values, analyze class imbalance, study outliers, and prepare decisions for the training pipeline.


## 1. Load Data

This cell finds the project root automatically, so the notebook works whether Jupyter starts in the repo root or inside `notebooks/`.


In [1]:
from pathlib import Path
import sys

import pandas as pd
from sklearn.model_selection import train_test_split


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for path in [current, *current.parents]:
        if (path / "data" / "raw" / "cs-training.csv").exists():
            return path
    raise FileNotFoundError("Could not find data/raw/cs-training.csv from the current notebook path.")


PROJECT_ROOT = find_project_root()
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

raw_df = pd.read_csv(RAW_DATA_DIR / "cs-training.csv")
raw_dictionary = pd.read_excel(RAW_DATA_DIR / "Data Dictionary.xls", header=None)

print(f"Project root: {PROJECT_ROOT}")
print(f"Raw training data: {raw_df.shape}")


Project root: C:\Users\shaya\OneDrive\Documents\mloops
Raw training data: (150000, 12)


## 2. Use The Data Dictionary

The dictionary is not model input. Use it to understand column meaning, data type, and README explanations.


In [2]:
df_dictionary = raw_dictionary.iloc[:, :3].copy()
df_dictionary.columns = df_dictionary.iloc[0]
df_dictionary = df_dictionary.iloc[1:].reset_index(drop=True)

df_dictionary


,NaN,NaN,NaN
0,Variable Name,Description,Type
1,SeriousDlqin2yrs,Person experienced 90 days past due delinquenc...,Y/N
2,RevolvingUtilizationOfUnsecuredLines,Total balance on credit cards and personal lin...,percentage
3,age,Age of borrower in years,integer
4,NumberOfTime30-59DaysPastDueNotWorse,Number of times borrower has been 30-59 days p...,integer
5,DebtRatio,"Monthly debt payments, alimony,living costs di...",percentage
6,MonthlyIncome,Monthly income,real
7,NumberOfOpenCreditLinesAndLoans,Number of Open loans (installment like car loa...,integer
8,NumberOfTimes90DaysLate,Number of times borrower has been 90 days or m...,integer
9,NumberRealEstateLoansOrLines,Number of mortgage and real estate loans inclu...,integer


## 3. Raw Dataset Checks

Before cleaning, check shape, missing values, target imbalance, and suspicious values.


In [3]:
display(raw_df.head())

summary = pd.DataFrame({
    "dtype": raw_df.dtypes.astype(str),
    "missing": raw_df.isna().sum(),
    "missing_pct": (raw_df.isna().mean() * 100).round(2),
    "n_unique": raw_df.nunique(),
})

display(summary)
print("Target distribution (%):")
display((raw_df["SeriousDlqin2yrs"].value_counts(normalize=True).sort_index() * 100).round(2))

print("Invalid ages:", int((raw_df["age"] <= 0).sum()))
print("Utilization > 1:", int((raw_df["RevolvingUtilizationOfUnsecuredLines"] > 1).sum()))

past_due_cols = [
    "NumberOfTime30-59DaysPastDueNotWorse",
    "NumberOfTime60-89DaysPastDueNotWorse",
    "NumberOfTimes90DaysLate",
]
for column in past_due_cols:
    print(column, raw_df[column].value_counts().sort_index().tail(8).to_dict())


,Unnamed: 0,SeriousDlqin2yrs,RevolvingUtilizationOfUnsecuredLines,age,NumberOfTime30-59DaysPastDueNotWorse,DebtRatio,MonthlyIncome,NumberOfOpenCreditLinesAndLoans,NumberOfTimes90DaysLate,NumberRealEstateLoansOrLines,NumberOfTime60-89DaysPastDueNotWorse,NumberOfDependents
0,1,1,0.766127,45,2,0.802982,9120.0,13,0,6,0,2.0
1,2,0,0.957151,40,0,0.121876,2600.0,4,0,0,0,1.0
2,3,0,0.658180,38,1,0.085113,3042.0,2,1,0,0,0.0
3,4,0,0.233810,30,0,0.036050,3300.0,5,0,0,0,0.0
4,5,0,0.907239,49,1,0.024926,63588.0,7,0,1,0,0.0


,dtype,missing,missing_pct,n_unique
Unnamed: 0,int64,0,0.00,150000
SeriousDlqin2yrs,int64,0,0.00,2
RevolvingUtilizationOfUnsecuredLines,float64,0,0.00,125728
age,int64,0,0.00,86
NumberOfTime30-59DaysPastDueNotWorse,int64,0,0.00,16
DebtRatio,float64,0,0.00,114194
MonthlyIncome,float64,29731,19.82,13594
NumberOfOpenCreditLinesAndLoans,int64,0,0.00,58
NumberOfTimes90DaysLate,int64,0,0.00,19
NumberRealEstateLoansOrLines,int64,0,0.00,28


Target distribution (%):


SeriousDlqin2yrs
0    93.32
1     6.68
Name: proportion, dtype: float64

Invalid ages: 1
Utilization > 1: 3321
NumberOfTime30-59DaysPastDueNotWorse {8: 25, 9: 12, 10: 4, 11: 1, 12: 2, 13: 1, 96: 5, 98: 264}
NumberOfTime60-89DaysPastDueNotWorse {5: 34, 6: 16, 7: 9, 8: 2, 9: 1, 11: 1, 96: 5, 98: 264}
NumberOfTimes90DaysLate {11: 5, 12: 2, 13: 4, 14: 2, 15: 2, 17: 1, 96: 5, 98: 264}


## 4. Leakage-Safe Cleaning

Split before fitting medians or clipping bounds. The preprocessor learns values from `X_train_raw` only, then reuses the same values for test data and future API requests.


In [4]:
from src.preprocessing import CreditRiskPreprocessor, TARGET_COLUMN

X_raw = raw_df.drop(columns=[TARGET_COLUMN])
y = raw_df[TARGET_COLUMN]

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

preprocessor = CreditRiskPreprocessor(clip_quantile=0.99)
X_train_clean = preprocessor.fit_transform(X_train_raw)
X_test_clean = preprocessor.transform(X_test_raw)
preprocessing_artifact = preprocessor.get_artifact().to_dict()

print("Train:", X_train_clean.shape, y_train.shape)
print("Test:", X_test_clean.shape, y_test.shape)
print("Missing values in train:", int(X_train_clean.isna().sum().sum()))
print("Missing values in test:", int(X_test_clean.isna().sum().sum()))


Train: (120000, 14) (120000,)
Test: (30000, 14) (30000,)
Missing values in train: 0
Missing values in test: 0


## 5. Inspect Cleaned Data

The cleaned data includes missing-value flags and outlier/sentinel handling features. These are useful because missing income or extreme delinquency codes may carry risk signal.


In [5]:
display(X_train_clean.head())
display(X_train_clean.describe().T)
preprocessing_artifact


,RevolvingUtilizationOfUnsecuredLines,age,NumberOfTime30_59DaysPastDueNotWorse,DebtRatio,MonthlyIncome,NumberOfOpenCreditLinesAndLoans,NumberOfTimes90DaysLate,NumberRealEstateLoansOrLines,NumberOfTime60_89DaysPastDueNotWorse,NumberOfDependents,MonthlyIncomeWasMissing,NumberOfDependentsWasMissing,AgeWasInvalid,PastDueExtremeCode
57836,0.114987,62.0,0.0,1841.000000,5390.0,5,0.0,1,0.0,2.0,1,0,0,0
132895,0.008705,73.0,0.0,0.498553,3800.0,6,0.0,1,0.0,0.0,0,0,0,0
27981,0.214501,32.0,0.0,0.211999,3716.0,8,0.0,0,0.0,2.0,0,0,0,0
37852,1.000000,60.0,0.0,118.000000,5390.0,5,0.0,0,0.0,0.0,1,0,0,0
103813,0.230493,60.0,0.0,1.017328,3000.0,10,0.0,1,0.0,0.0,0,0,0,0


,count,mean,std,min,25%,50%,75%,max
RevolvingUtilizationOfUnsecuredLines,120000.0,0.319799,0.351879,0.0,0.029593,0.153318,0.557832,1.094439
age,120000.0,52.288275,14.770503,21.0,41.000000,52.000000,63.000000,109.000000
NumberOfTime30_59DaysPastDueNotWorse,120000.0,0.268525,0.880460,0.0,0.000000,0.000000,0.000000,13.000000
DebtRatio,120000.0,315.909041,906.202216,0.0,0.175330,0.366194,0.860833,4977.000000
MonthlyIncome,120000.0,6151.521700,3919.512756,0.0,3900.000000,5390.000000,7400.000000,25000.000000
NumberOfOpenCreditLinesAndLoans,120000.0,8.465692,5.161693,0.0,5.000000,8.000000,11.000000,58.000000
NumberOfTimes90DaysLate,120000.0,0.120525,0.860942,0.0,0.000000,0.000000,0.000000,17.000000
NumberRealEstateLoansOrLines,120000.0,1.018167,1.133055,0.0,0.000000,1.000000,2.000000,54.000000
NumberOfTime60_89DaysPastDueNotWorse,120000.0,0.084742,0.568137,0.0,0.000000,0.000000,0.000000,11.000000
NumberOfDependents,120000.0,0.737592,1.107299,0.0,0.000000,0.000000,1.000000,20.000000


{'medians': {'MonthlyIncome': 5390.0, 'NumberOfDependents': 0.0, 'age': 52.0},
 'clip_upper_bounds': {'RevolvingUtilizationOfUnsecuredLines': 1.0944386634299974,
  'DebtRatio': 4977.0,
  'MonthlyIncome': 25000.0},
 'past_due_caps': {'NumberOfTime30_59DaysPastDueNotWorse': 13.0,
  'NumberOfTime60_89DaysPastDueNotWorse': 11.0,
  'NumberOfTimes90DaysLate': 17.0},
 'feature_columns': ['RevolvingUtilizationOfUnsecuredLines',
  'age',
  'NumberOfTime30_59DaysPastDueNotWorse',
  'DebtRatio',
  'MonthlyIncome',
  'NumberOfOpenCreditLinesAndLoans',
  'NumberOfTimes90DaysLate',
  'NumberRealEstateLoansOrLines',
  'NumberOfTime60_89DaysPastDueNotWorse',
  'NumberOfDependents',
  'MonthlyIncomeWasMissing',
  'NumberOfDependentsWasMissing',
  'AgeWasInvalid',
  'PastDueExtremeCode']}